# 03 — Silver RENAMU
## Transformación Bronze → Silver (Modelo EAV)

Procesa el **Registro Nacional de Municipalidades (RENAMU)** aplicando la estrategia **"Embudo EAV"**.

**El problema a resolver**: Entre 2012 y 2020, cada ZIP del RENAMU contiene decenas de módulos CSV (uno por formulario). A partir de 2021, es un único CSV plano. Si uniéramos los años directamente, el DataFrame de Silver tendría miles de columnas dispersas y llenas de nulos.

**La solución**: En lugar de unir tablas anchas, rotamos (Unpivot) cada archivo CSV módulo a módulo hacia el modelo **Entidad-Atributo-Valor (EAV)** antes de consolidar. El resultado siempre tiene el mismo esquema estrecho: `UBIGEO | ANO_RENAMU | MODULO | PREGUNTA | RESPUESTA`.

**Fuente:** `data/bronze/renamu/{año}/batch_*.json` (con columna `_source_file` por módulo)  
**Destino:** `data/silver/renamu/` (Parquet EAV particionado por ANO_RENAMU)

In [ ]:
import sys, time
from pathlib import Path
from functools import reduce

PROJECT_ROOT = Path().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from pyspark.sql import functions as F
from pyspark.sql.types import StringType

from src.paths import BRONZE, SILVER, CATEGORIAS_CSV, str_path, ensure_dirs
from src.spark_utils import get_spark
from src.transformations.common import clean_ghost_nulls
from core.audit.control_manager import ControlManager, ExecutionStatus

print("Imports OK")

In [ ]:
# ── Configuración ─────────────────────────────────────────────────────────────
BRONZE_BASE = BRONZE["renamu"]
SILVER_PATH = str_path(SILVER["renamu"])

# Columnas que identifican a la municipalidad (no son preguntas del cuestionario)
# Estas columnas se extraerán de cada módulo y se mantienen como dimensiones.
# Las búsqueda es case-insensitive.
ID_COL_CANDIDATES = [
    "ubigeo", "ccdd", "ccpp", "ccdi", "idmunici",
    "departamento", "provincia", "distrito",
    "ano_renamu", "_source_file",
]

# Tamaño de lote para el Unpivot (columnas por iteración de STACK)
# Reducir si se producen errores de memoria
CHUNK_SIZE = 80

print(f"Bronze source : {BRONZE_BASE}")
print(f"Silver target : {SILVER_PATH}")
ensure_dirs()

In [ ]:
spark = get_spark(app_name="MEF_Silver_RENAMU", memory="4g")

## PARTE 1: Lectura Bronze y Detección de Módulos

Los JSONs de Bronze contienen la columna `_source_file` que indica de qué módulo/CSV vino cada registro. Esto permite procesar cada módulo de forma independiente, evitando la explosión de columnas.

In [ ]:
if not BRONZE_BASE.exists() or not any(BRONZE_BASE.rglob("*.json")):
    raise FileNotFoundError(
        f"Sin datos Bronze RENAMU en {BRONZE_BASE}.\n"
        "Ejecutar primero: notebooks/00_bronze_ingestion.ipynb"
    )

# Leer todos los años disponibles
dfs_by_year = []
for year_dir in sorted(d for d in BRONZE_BASE.iterdir() if d.is_dir()):
    json_files = list(year_dir.rglob("*.json"))
    if not json_files:
        continue
    year_df = (
        spark.read.option("multiline", "false").json(str_path(year_dir))
        .withColumn("ANO_RENAMU", F.lit(year_dir.name))
    )
    dfs_by_year.append((year_dir.name, year_df))
    print(f"  Ano {year_dir.name}: {len(json_files)} archivo(s) JSON")

if not dfs_by_year:
    raise ValueError("No se encontraron datos JSON de RENAMU en Bronze.")

print(f"\nTotal años cargados: {len(dfs_by_year)}")

## PARTE 2: Función de Unpivot EAV por Módulo

Para cada año y cada módulo detectado (a través de `_source_file`), aplicamos:
1. Limpieza de ghost nulls
2. Reconstrucción de UBIGEO de 6 dígitos
3. Unpivot (STACK) de las columnas-pregunta en filas `PREGUNTA | RESPUESTA`

El resultado unificado siempre tiene el mismo esquema, sin importar el año.

In [ ]:
def detect_id_cols(df):
    """Detecta las columnas de identificación presentes en el DataFrame (case-insensitive)."""
    return [c for c in df.columns if c.lower() in ID_COL_CANDIDATES]


def build_ubigeo(df):
    """
    Intenta construir el UBIGEO de 6 dígitos a partir de las columnas disponibles.
    Prueba distintas combinaciones según la versión del formulario RENAMU.
    """
    cols_lower = {c.lower(): c for c in df.columns}

    # Caso 1: ya viene el ubigeo completo
    if "ubigeo" in cols_lower:
        return df.withColumn("UBIGEO", F.lpad(F.col(cols_lower["ubigeo"]).cast("string"), 6, "0"))

    # Caso 2: viene CCDD + CCPP + CCDI (formato modular antiguo)
    if all(k in cols_lower for k in ["ccdd", "ccpp", "ccdi"]):
        return df.withColumn("UBIGEO", F.concat(
            F.lpad(F.col(cols_lower["ccdd"]).cast("string"), 2, "0"),
            F.lpad(F.col(cols_lower["ccpp"]).cast("string"), 2, "0"),
            F.lpad(F.col(cols_lower["ccdi"]).cast("string"), 2, "0"),
        ))

    # Caso 3: no hay columna geográfica identificable — marcar como sin asignar
    return df.withColumn("UBIGEO", F.lit("000000"))


def unpivot_module(df, id_cols, question_cols, modulo_name):
    """
    Aplica el unpivot EAV a un subconjunto de columnas (un módulo),
    procesando en lotes de CHUNK_SIZE para evitar OutOfMemoryError.
    Retorna un DataFrame con esquema: UBIGEO | ANO_RENAMU | MODULO | PREGUNTA | RESPUESTA
    """
    # Castear todas las preguntas a string para homogeneizar
    for c in question_cols:
        df = df.withColumn(c, F.col(c).cast(StringType()))

    result_chunks = []
    for i in range(0, len(question_cols), CHUNK_SIZE):
        chunk = question_cols[i:i + CHUNK_SIZE]
        n = len(chunk)
        pairs = ", ".join([f"'{c}', `{c}`" for c in chunk])
        stack_expr = f"STACK({n}, {pairs}) AS (PREGUNTA, RESPUESTA)"

        id_exprs = [F.col(c) for c in id_cols]
        chunk_df = (
            df.select(*id_exprs, F.expr(stack_expr))
            .filter(F.col("RESPUESTA").isNotNull() & (F.col("RESPUESTA") != ""))
            .withColumn("MODULO", F.lit(modulo_name))
        )
        result_chunks.append(chunk_df)

    if not result_chunks:
        return None

    return reduce(lambda a, b: a.union(b), result_chunks)


print(f"Funciones EAV definidas. Chunk size: {CHUNK_SIZE} columnas/lote.")

## PARTE 3: Procesamiento Año por Año → Módulo por Módulo

Para cada año y cada módulo (identificado por `_source_file`), se aplica el Unpivot EAV. Los resultados se apilan verticamente. Esto garantiza uniformidad del esquema entre 2012 y 2024.

In [ ]:
control = ControlManager(pipeline_name="silver_renamu")
execution = control.start(input_parameters={"source": "renamu", "estrategia": "EAV"})

all_eav = []
start_time = time.time()

try:
    for year_str, raw_df in dfs_by_year:
        print(f"\n--- Procesando ano {year_str} ---")

        # Limpiar ghost nulls
        clean_df = clean_ghost_nulls(raw_df)

        # Construir UBIGEO normalizado
        clean_df = build_ubigeo(clean_df)

        # Detectar si hay columna _source_file (años modulares: 2012-2020)
        if "_source_file" in [c.lower() for c in clean_df.columns]:
            # Obtener módulos únicos presentes en este año
            modulos = [row[0] for row in clean_df.select("_source_file").distinct().collect()]
            print(f"  Modulos detectados: {len(modulos)}")

            for modulo in sorted(modulos):
                mod_df = clean_df.filter(F.col("_source_file") == modulo)

                # Columnas de identificación y preguntas
                id_cols = ["UBIGEO", "ANO_RENAMU"]
                question_cols = [
                    c for c in mod_df.columns
                    if c.lower() not in ID_COL_CANDIDATES and c not in id_cols
                ]

                if not question_cols:
                    print(f"    [{modulo}] Sin columnas-pregunta, omitiendo.")
                    continue

                eav_df = unpivot_module(mod_df, id_cols, question_cols, modulo)
                if eav_df:
                    all_eav.append(eav_df)
                    print(f"    [{modulo}] {len(question_cols)} preguntas rotadas.")

        else:
            # Año con CSV único (2021+): tratar todo como un solo módulo
            id_cols = ["UBIGEO", "ANO_RENAMU"]
            question_cols = [
                c for c in clean_df.columns
                if c.lower() not in ID_COL_CANDIDATES and c not in id_cols
            ]
            print(f"  Formato CSV unico: {len(question_cols)} columnas-pregunta.")

            eav_df = unpivot_module(clean_df, id_cols, question_cols, "UNICO")
            if eav_df:
                all_eav.append(eav_df)

    if not all_eav:
        raise ValueError("No se generó ningún DataFrame EAV. Revisar datos Bronze.")

    print(f"\nConsolidando {len(all_eav)} DataFrame(s) EAV...")
    final_df = reduce(lambda a, b: a.unionByName(b, allowMissingColumns=True), all_eav)

    # Enriquecer con categoría municipal si existe el CSV de referencia
    if CATEGORIAS_CSV.exists():
        cat_df = (
            spark.read.option("header", "true").csv(str_path(CATEGORIAS_CSV))
            .withColumn("UBIGEO_REF", F.lpad(F.col("UBIGEO").cast("string"), 6, "0"))
            .select("UBIGEO_REF", "CATEGORIA")
            .dropDuplicates(["UBIGEO_REF"])
        )
        final_df = (
            final_df.join(cat_df, final_df.UBIGEO == cat_df.UBIGEO_REF, "left")
            .drop("UBIGEO_REF")
            .fillna({"CATEGORIA": "SIN_CATEGORIA"})
        )
        print("Join con categorias municipales aplicado.")

    # Escribir Silver particionado por año
    (
        final_df.write
        .mode("overwrite")
        .partitionBy("ANO_RENAMU")
        .format("parquet")
        .save(SILVER_PATH)
    )

    elapsed = time.time() - start_time
    n_final = final_df.count()
    control.end(
        status=ExecutionStatus.SUCCESS,
        output_summary={"total_filas_eav": n_final, "anos": len(dfs_by_year), "elapsed_sec": round(elapsed, 1)}
    )
    print(f"\nSilver RENAMU (EAV) completado en {elapsed:.1f}s")
    print(f"   Total filas EAV: {n_final:,}")

except Exception as e:
    control.log_error("SilverRenamuError", str(e))
    control.end(status=ExecutionStatus.FAILED, output_summary={"error": str(e)})
    raise

In [ ]:
# ── Verificación Final ────────────────────────────────────────────────────────
df_check = spark.read.parquet(SILVER_PATH)
print(f"Verificacion Silver RENAMU (EAV):")
print(f"  Total filas    : {df_check.count():,}")
print(f"  Columnas       : {df_check.columns}")
print(f"  Anos presentes :")
df_check.groupBy("ANO_RENAMU").count().orderBy("ANO_RENAMU").show(20)
print(f"  Top 10 preguntas mas frecuentes:")
df_check.groupBy("PREGUNTA").count().orderBy("count", ascending=False).show(10, truncate=False)